# EDA y revisión del clasificador – NTP 900.058
Revisa el **dataset** (`data basurita`) y el **modelo entrenado** (`clasificador.pt`) para saber cuánto confiar en sus resultados:

1. **EDA**: cuántas fotos hay por color, split y origen; tamaños, brillo, nitidez y fondos.
2. **Atajo del fondo**: ¿se puede adivinar el color mirando solo el borde de la foto?
3. **Fuga de datos**: fotos casi iguales repartidas entre train y val/test.
4. **Evaluación**: acierto por color, por origen y **sin las fotos casi duplicadas** (acierto "limpio").
5. **¿Hace trampa?**: acierto tapando el objeto (solo fondo) y mapas **Grad-CAM** de dónde mira el modelo.
6. **Test con tu cámara** (si existe `data basurita/test_camara/<Color>/`).
7. **Curvas del entrenamiento** (si existe `data basurita/runs_clasificador/`).
8. **Resumen con semáforo**.

**Cómo usarlo:** `Entorno de ejecución → Ejecutar todo`. Solo tienes que autorizar el acceso a Drive.
Funciona con CPU (~15–30 min); con GPU es algo más rápido. **No modifica el dataset ni el modelo.**
Todo se guarda en `data basurita/revision/`.

## 1. Instalar

In [ ]:
!pip install -q "ultralytics>=8.3,<8.4" imagehash pillow-heif

## 2. Configuración y conectar Drive

In [ ]:
import os, glob, shutil, zipfile, json, time
from concurrent.futures import ThreadPoolExecutor

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/data basurita"
    TRABAJO = "/content"
except ImportError:  # ejecucion fuera de Colab (pruebas)
    DRIVE_DIR = os.environ.get("DRIVE_DIR", "data basurita")
    TRABAJO = os.environ.get("TRABAJO", "/tmp")

ZIP = f"{DRIVE_DIR}/dataset_ntp900058.zip"
MODELO = f"{DRIVE_DIR}/modelos/clasificador.pt"
TEST_CAMARA = f"{DRIVE_DIR}/test_camara"     # opcional: test_camara/<Color>/fotos
RUNS = f"{DRIVE_DIR}/runs_clasificador"      # opcional: curvas del entrenamiento
REV = f"{DRIVE_DIR}/revision"                # aqui se guarda todo
DATASET = f"{TRABAJO}/dataset_eda"           # copia local del zip

COLORES = ["Azul", "Amarillo", "Blanco", "Gris", "Marron", "Rojo", "Negro"]
NOMBRES = {  # carpeta del dataset -> clase del modelo
    "Azul": "azul_papel_carton", "Amarillo": "amarillo_metales", "Blanco": "blanco_plasticos",
    "Gris": "gris_vidrio", "Marron": "marron_organicos", "Rojo": "rojo_peligrosos", "Negro": "negro_generales",
}
COLOR_GRAFICO = {"Azul": "#1f77b4", "Amarillo": "#e6b800", "Blanco": "#bbbbbb", "Gris": "#7f7f7f",
                 "Marron": "#8c564b", "Rojo": "#d62728", "Negro": "#222222"}
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".heic"}
SEMILLA = 42
UMBRAL_DUP = 6        # bits distintos (de 64) para considerar dos fotos "casi iguales"
MAX_ANTITRAMPA = 700  # fotos de test usadas en las pruebas de fondo y Grad-CAM

import numpy as np, pandas as pd, cv2, torch, imagehash
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
import pillow_heif
pillow_heif.register_heif_opener()
try:
    from IPython.display import display
except ImportError:
    display = print

os.makedirs(REV, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
RESUMEN = {}  # numeros clave para el resumen final
rng = np.random.default_rng(SEMILLA)


def guardar_figura(nombre):
    plt.savefig(f"{REV}/{nombre}", dpi=110, bbox_inches="tight")
    plt.show()
    plt.close("all")


print("Drive:", DRIVE_DIR, "| dispositivo:", "GPU" if DEVICE == 0 else "CPU")
print("Resultados en:", REV)

## 3. Cargar el dataset y el modelo
Usa el mismo `.zip` con el que se entrenó (o la carpeta `splits` si no hay zip).

In [ ]:
from ultralytics import YOLO

shutil.rmtree(DATASET, ignore_errors=True)
if os.path.exists(ZIP):
    zipfile.ZipFile(ZIP).extractall(DATASET)
elif os.path.isdir(f"{DRIVE_DIR}/splits"):
    shutil.copytree(f"{DRIVE_DIR}/splits", DATASET)
else:
    raise SystemExit(f"No encontre {ZIP} ni {DRIVE_DIR}/splits. Ejecuta antes dataset_ntp900058_splits.ipynb.")

# Carpeta que contiene train/val/test (por si el zip trae una carpeta extra)
BASE = next((r for r, dirs, _ in os.walk(DATASET) if {"train", "val", "test"} <= set(dirs)), None)
if BASE is None:
    raise SystemExit("El dataset no tiene las carpetas train/val/test.")

filas, extrañas = [], set()
for split in ("train", "val", "test"):
    for carpeta in sorted(os.listdir(f"{BASE}/{split}")):
        color = carpeta if carpeta in COLORES else {v: k for k, v in NOMBRES.items()}.get(carpeta)
        if color is None:
            extrañas.add(carpeta)
            continue
        for f in sorted(os.listdir(f"{BASE}/{split}/{carpeta}")):
            if os.path.splitext(f)[1].lower() in EXT:
                filas.append({"split": split, "color": color, "archivo": f,
                              "origen": "propia" if f.startswith("propia_") else "descargada",
                              "ruta": f"{BASE}/{split}/{carpeta}/{f}"})
df = pd.DataFrame(filas)
if extrañas:
    print("AVISO: carpetas que no son un color (se ignoran):", sorted(extrañas))
print(f"{len(df)} fotos cargadas")

if not os.path.exists(MODELO):
    alternativas = sorted(glob.glob(f"{RUNS}/etapa2*/weights/best.pt") + glob.glob(f"{RUNS}/etapa1*/weights/best.pt"))
    if not alternativas:
        raise SystemExit(f"No encontre el modelo en {MODELO}.")
    MODELO = alternativas[0]
modelo = YOLO(MODELO)
A_COLOR = {v: k for k, v in NOMBRES.items()}  # clase del modelo -> color corto
print("Modelo:", MODELO)
print("Clases del modelo:", list(modelo.names.values()))
faltan = set(COLORES) - {A_COLOR.get(n, n) for n in modelo.names.values()}
if faltan:
    print("AVISO: el modelo no tiene estas clases:", faltan)

## 4. Analizar cada foto
Calcula tamaño, brillo, nitidez, color del **borde** (≈ fondo) y una huella perceptual (*pHash*)
para encontrar fotos casi iguales. Tarda unos minutos.

In [ ]:
def analizar(ruta):
    try:
        with Image.open(ruta) as im:
            im = im.convert("RGB")
            huella = int(str(imagehash.phash(im)), 16)
            a = np.asarray(im)
    except Exception as e:
        return {"error": str(e)}
    h, w = a.shape[:2]
    gris = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY)
    b = max(2, int(min(h, w) * 0.1))  # franja del borde: 10% de cada lado
    borde = np.concatenate([a[:b].reshape(-1, 3), a[-b:].reshape(-1, 3),
                            a[:, :b].reshape(-1, 3), a[:, -b:].reshape(-1, 3)]).astype(np.float32)
    return {"ancho": w, "alto": h, "brillo": float(gris.mean()),
            "nitidez": float(cv2.Laplacian(gris, cv2.CV_64F).var()),
            "borde_r": float(borde[:, 0].mean()), "borde_g": float(borde[:, 1].mean()),
            "borde_b": float(borde[:, 2].mean()), "borde_std": float(borde.std()),
            "huella": huella, "error": None}

t0 = time.time()
with ThreadPoolExecutor(8) as ex:
    datos = list(ex.map(analizar, df["ruta"]))
df = pd.concat([df.reset_index(drop=True), pd.DataFrame(datos)], axis=1)
malas = df[df["error"].notna()]
if len(malas):
    print(f"AVISO: {len(malas)} fotos no se pudieron abrir (se excluyen):")
    print(malas[["split", "color", "archivo", "error"]].head(20).to_string())
df = df[df["error"].isna()].reset_index(drop=True)
print(f"{len(df)} fotos analizadas en {time.time() - t0:.0f} s")

## 5. EDA: cuántas fotos hay

In [ ]:
tabla = pd.crosstab(df["color"], df["split"]).reindex(index=COLORES, columns=["train", "val", "test"], fill_value=0)
tabla["total"] = tabla.sum(axis=1)
tabla["% del total"] = (100 * tabla["total"] / tabla["total"].sum()).round(1)
tabla.loc["TOTAL"] = tabla.sum(numeric_only=True)
tabla.loc["TOTAL", "% del total"] = 100.0
print("Fotos por color y split")
display(tabla)

origen = pd.crosstab(df["color"], df["origen"]).reindex(COLORES, fill_value=0)
print("\nFotos por origen (descargadas = Garbage V2 + TrashBox; propias = tus fotos del Drive)")
display(origen)
tabla.to_csv(f"{REV}/eda_conteo.csv")
origen.to_csv(f"{REV}/eda_origen.csv")

conteo = tabla.loc[COLORES, "total"]
RESUMEN["fotos_total"] = int(conteo.sum())
RESUMEN["desbalance_max_min"] = round(float(conteo.max() / max(conteo.min(), 1)), 2)
print(f"\nDesbalance (color con mas fotos / color con menos): {RESUMEN['desbalance_max_min']}x")
if RESUMEN["desbalance_max_min"] > 2:
    print("AVISO: hay colores con mas del doble de fotos que otros; el modelo puede inclinarse hacia ellos.")

fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
tabla.loc[COLORES, ["train", "val", "test"]].plot.bar(stacked=True, ax=ejes[0], color=["#4c72b0", "#dd8452", "#55a868"])
ejes[0].set_title("Fotos por color y split"); ejes[0].set_xlabel("")
origen.plot.bar(stacked=True, ax=ejes[1], color=["#8da0cb", "#fc8d62"])
ejes[1].set_title("Fotos por origen"); ejes[1].set_xlabel("")
guardar_figura("eda_conteo.png")

## 6. EDA: tamaño, brillo y nitidez por color

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, titulo in zip(ejes, ["brillo", "nitidez", "ancho"],
                           ["Brillo medio (0-255)", "Nitidez (varianza del Laplaciano, escala log)", "Ancho (px)"]):
    datos_col = [df.loc[df["color"] == c, col] for c in COLORES]
    caja = ax.boxplot(datos_col, patch_artist=True, showfliers=False)
    ax.set_xticks(range(1, len(COLORES) + 1), COLORES)
    for parche, c in zip(caja["boxes"], COLORES):
        parche.set_facecolor(COLOR_GRAFICO[c]); parche.set_alpha(0.6)
    ax.set_title(titulo); ax.tick_params(axis="x", rotation=45)
    if col == "nitidez":
        ax.set_yscale("log")
guardar_figura("eda_brillo_nitidez_tamano.png")

stats = df.groupby("color")[["brillo", "nitidez", "ancho", "alto"]].median().reindex(COLORES).round(1)
print("Medianas por color")
display(stats)
borrosas = (df["nitidez"] < 20).mean()
pequenas = ((df["ancho"] < 150) | (df["alto"] < 150)).mean()
print(f"Fotos muy borrosas (nitidez < 20): {borrosas:.1%} | muy pequenas (< 150 px): {pequenas:.1%}")
RESUMEN["fotos_borrosas_pct"] = round(100 * borrosas, 1)

## 7. EDA: muestras y color del fondo
Si cada color tuviera un fondo típico (por ejemplo, las pilas siempre sobre blanco),
el modelo podría aprender el fondo en vez del material.

In [ ]:
fig, ejes = plt.subplots(len(COLORES), 6, figsize=(13, 2.3 * len(COLORES)))
for fila, c in enumerate(COLORES):
    sub = df[(df["color"] == c) & (df["split"] == "train")]
    muestras = sub.sample(min(6, len(sub)), random_state=SEMILLA)["ruta"].tolist() if len(sub) else []
    for col in range(6):
        ax = ejes[fila][col]; ax.axis("off")
        if col < len(muestras):
            ax.imshow(Image.open(muestras[col]).convert("RGB"))
        if col == 0:
            ax.set_title(c, loc="left", fontsize=12, fontweight="bold")
guardar_figura("eda_muestras.png")

fondo = df.groupby("color")[["borde_r", "borde_g", "borde_b", "borde_std"]].mean().reindex(COLORES)
fig, ax = plt.subplots(figsize=(10, 1.6))
for i, c in enumerate(COLORES):
    rgb = fondo.loc[c, ["borde_r", "borde_g", "borde_b"]].to_numpy() / 255
    ax.add_patch(plt.Rectangle((i, 0), 0.9, 1, color=np.clip(rgb, 0, 1)))
    ax.text(i + 0.45, -0.25, c, ha="center", fontsize=10)
ax.set_xlim(0, len(COLORES)); ax.set_ylim(-0.5, 1); ax.axis("off")
ax.set_title("Color promedio del borde de las fotos (aprox. el fondo)")
guardar_figura("eda_color_fondo.png")
display(fondo.round(1))

## 8. ¿Se puede adivinar el color solo con el fondo?
Un modelo muy simple (regresión logística) intenta adivinar el color usando **solo** el color del borde,
el brillo y la variación del borde, sin mirar el objeto. Si acierta mucho más que el azar, el dataset
tiene un **atajo de fondo** que el clasificador también podría estar usando.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rasgos = ["borde_r", "borde_g", "borde_b", "borde_std", "brillo"]
tr, te = df[df["split"] == "train"], df[df["split"] == "test"]
azar = te["color"].value_counts(normalize=True).max() if len(te) else 1 / len(COLORES)
if tr["color"].nunique() > 1 and len(te):
    simple = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(tr[rasgos], tr["color"])
    acierto_fondo = float((simple.predict(te[rasgos]) == te["color"]).mean())
else:
    acierto_fondo = float("nan")
RESUMEN["atajo_fondo_acierto"] = round(100 * acierto_fondo, 1)
RESUMEN["azar_pct"] = round(100 * azar, 1)
print(f"Acierto adivinando solo con el fondo: {acierto_fondo:.1%}  (azar: {azar:.1%})")
if acierto_fondo > 0.5:
    print("ALERTA: el fondo dice mucho del color. Conviene fotos del mismo color con fondos distintos.")
elif acierto_fondo > 0.35:
    print("Atajo moderado: el fondo da algunas pistas. Revisa las pruebas del paso 11.")
else:
    print("Bien: el fondo casi no permite adivinar el color.")

## 9. Fuga de datos: fotos casi iguales entre train y val/test
Compara la huella perceptual de cada foto de val/test con **todas** las de train.
Si la diferencia es de pocos bits (≤ `UMBRAL_DUP`), son casi la misma foto: el modelo ya la "vio".

In [ ]:
TABLA_BITS = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def distancias_minimas(consulta, base, bloque=256):
    """Para cada huella de 'consulta', la menor distancia (bits) a 'base' y su indice."""
    dist, idx = np.full(len(consulta), 64, np.int16), np.zeros(len(consulta), np.int64)
    if len(base) == 0:
        return dist, idx
    for i in range(0, len(consulta), bloque):
        x = np.bitwise_xor(consulta[i:i + bloque, None], base[None, :])
        d = TABLA_BITS[x.view(np.uint8)].reshape(x.shape[0], x.shape[1], 8).sum(-1)
        idx[i:i + bloque] = d.argmin(1)
        dist[i:i + bloque] = d.min(1)
    return dist, idx

huellas = np.array([np.uint64(h) for h in df["huella"]], dtype=np.uint64)
es_train = (df["split"] == "train").to_numpy()
pos_train = np.flatnonzero(es_train)
evaluar = np.flatnonzero(~es_train)
d, i = distancias_minimas(huellas[evaluar], huellas[pos_train])
df["dist_train"] = 64
df.loc[evaluar, "dist_train"] = d
df["gemelo_train"] = ""
df.loc[evaluar, "gemelo_train"] = df.loc[pos_train[i], "ruta"].to_numpy()
df["casi_dup"] = df["dist_train"] <= UMBRAL_DUP

dup = df[~es_train].groupby(["split", "color"])["casi_dup"].mean().unstack(0).reindex(COLORES)
print(f"% de fotos de val/test con una foto casi igual en train (≤ {UMBRAL_DUP} bits):")
display((100 * dup).round(1))
RESUMEN["casi_dup_test_pct"] = round(100 * float(df.loc[df["split"] == "test", "casi_dup"].mean()), 1)
RESUMEN["casi_dup_val_pct"] = round(100 * float(df.loc[df["split"] == "val", "casi_dup"].mean()), 1)
print(f"\nTotal test: {RESUMEN['casi_dup_test_pct']}% | total val: {RESUMEN['casi_dup_val_pct']}%")

pares = df[df["casi_dup"]].sort_values("dist_train")
pares[["split", "color", "archivo", "dist_train", "gemelo_train"]].to_csv(f"{REV}/casi_duplicados.csv", index=False)
if len(pares):
    n = min(8, len(pares))
    fig, ejes = plt.subplots(2, n, figsize=(2.2 * n, 4.6), squeeze=False)
    for k, (_, f) in enumerate(pares.head(n).iterrows()):
        for fila, ruta, titulo in ((0, f["ruta"], f"{f['split']} · {f['color']}"), (1, f["gemelo_train"], f"train · {f['dist_train']} bits")):
            ax = ejes[fila][k]; ax.imshow(Image.open(ruta).convert("RGB")); ax.axis("off"); ax.set_title(titulo, fontsize=8)
    plt.suptitle("Arriba: foto de val/test · Abajo: su casi gemela en train")
    guardar_figura("casi_duplicados.png")
else:
    print("No se encontraron fotos casi iguales entre train y val/test.")

## 10. Evaluación del modelo en val y test
Además del acierto normal, calcula el acierto **limpio**: solo con las fotos de test que **no** tienen
una casi gemela en train. Es la cifra más honesta del dataset.

In [ ]:
def predecir(fuentes, lote=64):
    """fuentes: rutas o imagenes BGR. Devuelve (color predicho, confianza)."""
    colores, confs = [], []
    for k in range(0, len(fuentes), lote):
        for r in modelo.predict(fuentes[k:k + lote], imgsz=224, device=DEVICE, verbose=False):
            nombre = r.names[r.probs.top1]
            colores.append(A_COLOR.get(nombre, nombre)); confs.append(float(r.probs.top1conf))
    return colores, confs

t0 = time.time()
ev = df[df["split"].isin(["val", "test"])].copy()
ev["pred"], ev["conf"] = predecir(ev["ruta"].tolist())
ev["acierto"] = ev["pred"] == ev["color"]
print(f"{len(ev)} fotos evaluadas en {time.time() - t0:.0f} s\n")

te = ev[ev["split"] == "test"]
limpio = te[~te["casi_dup"]]
filas = []
for c in COLORES:
    s, l = te[te["color"] == c], limpio[limpio["color"] == c]
    filas.append({"color": c, "fotos test": len(s), "acierto": s["acierto"].mean(),
                  "acierto limpio": l["acierto"].mean() if len(l) else np.nan, "fotos limpias": len(l)})
por_color = pd.DataFrame(filas).set_index("color")
print("Acierto por color en TEST (acierto limpio = sin fotos casi duplicadas de train)")
display((por_color.style.format({"acierto": "{:.1%}", "acierto limpio": "{:.1%}"})
         if hasattr(por_color, "style") and display is not print else por_color.round(3)))
por_color.to_csv(f"{REV}/acierto_por_color.csv")

RESUMEN["val_acierto"] = round(100 * float(ev.loc[ev["split"] == "val", "acierto"].mean()), 1)
RESUMEN["test_acierto"] = round(100 * float(te["acierto"].mean()), 1)
RESUMEN["test_acierto_limpio"] = round(100 * float(limpio["acierto"].mean()), 1) if len(limpio) else None
print(f"\nval: {RESUMEN['val_acierto']}% | test: {RESUMEN['test_acierto']}% | "
      f"test limpio: {RESUMEN['test_acierto_limpio']}% ({len(limpio)} de {len(te)} fotos)")

por_origen = te.groupby("origen")["acierto"].agg(["mean", "count"]).rename(columns={"mean": "acierto", "count": "fotos"})
print("\nAcierto en test por origen:")
display(por_origen.round(3))
if "propia" in por_origen.index:
    RESUMEN["test_acierto_propias"] = round(100 * float(por_origen.loc["propia", "acierto"]), 1)

matriz = pd.crosstab(te["color"], te["pred"], normalize="index").reindex(index=COLORES, columns=COLORES, fill_value=0)
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(matriz.to_numpy(), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(COLORES)), COLORES, rotation=45); ax.set_yticks(range(len(COLORES)), COLORES)
for a in range(len(COLORES)):
    for b in range(len(COLORES)):
        v = matriz.iloc[a, b]
        if v >= 0.005:
            ax.text(b, a, f"{v:.0%}", ha="center", va="center", fontsize=9, color="white" if v > 0.5 else "black")
ax.set_xlabel("Predicho"); ax.set_ylabel("Real"); ax.set_title("Matriz de confusión (test, % por fila)")
guardar_figura("matriz_confusion_test.png")

errores = te[~te["acierto"]].sort_values("conf", ascending=False)
ev.drop(columns=["huella"]).to_csv(f"{REV}/predicciones_val_test.csv", index=False)
if len(errores):
    n = min(12, len(errores))
    fig, ejes = plt.subplots(2, (n + 1) // 2, figsize=(2.4 * ((n + 1) // 2), 5.4), squeeze=False)
    for ax in ejes.flat:
        ax.axis("off")
    for ax, (_, f) in zip(ejes.flat, errores.head(n).iterrows()):
        ax.imshow(Image.open(f["ruta"]).convert("RGB"))
        ax.set_title(f"real {f['color']}\npred {f['pred']} {f['conf']:.0%}", fontsize=8, color="#b00")
    plt.suptitle("Errores con más confianza (test)")
    guardar_figura("errores_test.png")

## 11. ¿Hace trampa? Tapando el objeto
Se repite la predicción en test con dos versiones de cada foto:
- **Solo fondo**: el centro (60%) se tapa con gris. Si aún acierta mucho, mira el fondo.
- **Solo centro**: se recorta el centro (60%). Debería seguir acertando.

Además se prueban imágenes sin objeto (ruido y colores lisos): el modelo siempre elige un color,
pero idealmente con poca confianza.

In [ ]:
muestra = te.sample(min(MAX_ANTITRAMPA, len(te)), random_state=SEMILLA)

def solo_fondo(img):
    h, w = img.shape[:2]
    out = img.copy()
    out[int(h * .2):int(h * .8), int(w * .2):int(w * .8)] = 114
    return out

def solo_centro(img):
    h, w = img.shape[:2]
    return img[int(h * .2):int(h * .8), int(w * .2):int(w * .8)].copy()

imagenes = [cv2.cvtColor(np.asarray(Image.open(r).convert("RGB")), cv2.COLOR_RGB2BGR) for r in muestra["ruta"]]
pf, _ = predecir([solo_fondo(i) for i in imagenes])
pc, _ = predecir([solo_centro(i) for i in imagenes])
real = muestra["color"].to_numpy()
RESUMEN["solo_fondo_acierto"] = round(100 * float((np.array(pf) == real).mean()), 1)
RESUMEN["solo_centro_acierto"] = round(100 * float((np.array(pc) == real).mean()), 1)
RESUMEN["foto_completa_acierto_muestra"] = round(100 * float(muestra["acierto"].mean()), 1)
print(f"Foto completa: {RESUMEN['foto_completa_acierto_muestra']}% | solo centro: {RESUMEN['solo_centro_acierto']}% "
      f"| solo fondo: {RESUMEN['solo_fondo_acierto']}% (azar: {RESUMEN['azar_pct']}%)")

pruebas = {"ruido": rng.integers(0, 256, (480, 640, 3), dtype=np.uint8),
           "gris liso": np.full((480, 640, 3), 128, np.uint8),
           "blanco liso": np.full((480, 640, 3), 235, np.uint8),
           "negro liso": np.full((480, 640, 3), 20, np.uint8)}
pv, cv_ = predecir(list(pruebas.values()))
print("\nImágenes sin objeto (lo ideal: confianza baja):")
for (nombre, _), p, cf in zip(pruebas.items(), pv, cv_):
    aviso = "  <-- pasaria el filtro del plan B (70%)" if cf >= 0.7 else ""
    print(f"  {nombre:<12} -> {p:<9} {cf:.0%}{aviso}")
RESUMEN["sin_objeto_max_conf"] = round(100 * max(cv_), 1)

## 12. Grad-CAM: dónde mira el modelo
El mapa de calor muestra las zonas que más influyeron en la decisión (rojo = mucho).
Lo ideal es que el calor esté **sobre el objeto**, no en el fondo.
`foco en el centro` = parte del calor que cae en la zona central (si fuera uniforme sería 25%).

In [ ]:
from ultralytics.data.augment import classify_transforms

red = YOLO(MODELO).model.float().eval()
for p in red.parameters():
    p.requires_grad_(False)
CAPA = red.model[-2]  # ultima capa de la base (C2PSA)
transformar = classify_transforms(224)
INDICE = {A_COLOR.get(n, n): k for k, n in red.names.items()}

def gradcam(ruta=None, img_bgr=None):
    """Devuelve (imagen 224x224 RGB, mapa 224x224 en 0-1, color predicho, confianza)."""
    im = Image.open(ruta).convert("RGB") if ruta is not None else Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    x = transformar(im).unsqueeze(0).requires_grad_(True)
    guardado = {}
    def gancho(modulo, entrada, salida):
        salida.retain_grad(); guardado["a"] = salida
    h = CAPA.register_forward_hook(gancho)
    prob, logits = red(x)
    h.remove()
    k = int(prob.argmax(1))
    logits[0, k].backward()
    a, g = guardado["a"], guardado["a"].grad
    mapa = torch.relu((g.mean((2, 3), keepdim=True) * a).sum(1))[0].detach()
    mapa = torch.nn.functional.interpolate(mapa[None, None], size=(224, 224), mode="bilinear")[0, 0].numpy()
    mapa = mapa / (mapa.max() + 1e-8)
    vista = (x[0].detach().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    nombre = red.names[k]
    return vista, mapa, A_COLOR.get(nombre, nombre), float(prob[0, k])

def foco_centro(mapa):
    return float(mapa[56:168, 56:168].sum() / (mapa.sum() + 1e-8))

def superponer(vista, mapa):
    calor = cv2.applyColorMap((mapa * 255).astype(np.uint8), cv2.COLORMAP_JET)[:, :, ::-1]
    return (0.55 * vista + 0.45 * calor).astype(np.uint8)

def grilla_gradcam(rutas_titulos, titulo, archivo, columnas=6):
    filas_n = (len(rutas_titulos) + columnas - 1) // columnas
    fig, ejes = plt.subplots(filas_n, columnas, figsize=(2.4 * columnas, 2.7 * filas_n), squeeze=False)
    for ax in ejes.flat:
        ax.axis("off")
    for ax, (ruta, real) in zip(ejes.flat, rutas_titulos):
        vista, mapa, pred, conf = gradcam(ruta)
        ax.imshow(superponer(vista, mapa))
        ax.set_title(f"real {real}\npred {pred} {conf:.0%}", fontsize=8, color="black" if pred == real else "#b00")
    plt.suptitle(titulo)
    guardar_figura(archivo)

t0 = time.time()
focos = [foco_centro(gradcam(r)[1]) for r in muestra["ruta"].head(300)]
RESUMEN["gradcam_foco_centro_pct"] = round(100 * float(np.mean(focos)), 1)
print(f"Foco en el centro (promedio de {len(focos)} fotos): {RESUMEN['gradcam_foco_centro_pct']}% "
      f"(uniforme = 25%) · {time.time() - t0:.0f} s")

aciertos = [(r, c) for c in COLORES for r in te[(te["color"] == c) & te["acierto"]]["ruta"].head(2)]
grilla_gradcam(aciertos, "Grad-CAM: aciertos (2 por color)", "gradcam_aciertos.png")
if len(errores):
    grilla_gradcam(list(zip(errores["ruta"].head(12), errores["color"].head(12))),
                   "Grad-CAM: errores con más confianza", "gradcam_errores.png")

## 13. Test con tu cámara (opcional)
Toma ~20 fotos por color con tu webcam (`python src/detector_residuos.py --recolectar Blanco`, tecla `c`)
o con el celular, de objetos que **nunca** se usaron para entrenar, y súbelas a
`data basurita/test_camara/<Color>/` (Azul, Amarillo, Blanco, Gris, Marron, Rojo, Negro).
Si la carpeta no existe, este paso se salta.

In [ ]:
camara = []
if os.path.isdir(TEST_CAMARA):
    for c in COLORES:
        for f in sorted(glob.glob(f"{TEST_CAMARA}/{c}/**/*", recursive=True)):
            if os.path.splitext(f)[1].lower() in EXT:
                camara.append({"color": c, "ruta": f})
if camara:
    cam = pd.DataFrame(camara)
    # las fotos HEIC u otras se convierten a BGR para el modelo
    imgs = [cv2.cvtColor(np.asarray(Image.open(r).convert("RGB")), cv2.COLOR_RGB2BGR) for r in cam["ruta"]]
    cam["pred"], cam["conf"] = predecir(imgs)
    cam["acierto"] = cam["pred"] == cam["color"]
    RESUMEN["camara_acierto"] = round(100 * float(cam["acierto"].mean()), 1)
    RESUMEN["camara_fotos"] = len(cam)
    print(f"Acierto con tu cámara: {RESUMEN['camara_acierto']}% ({len(cam)} fotos)")
    display(cam.groupby("color")["acierto"].agg(["mean", "count"]).reindex(COLORES).round(3))
    cam.to_csv(f"{REV}/test_camara.csv", index=False)
    elegidas = [(r, c) for c in COLORES for r in cam[cam["color"] == c]["ruta"].head(2)]
    grilla_gradcam(elegidas, "Grad-CAM con fotos de tu cámara", "gradcam_camara.png")
else:
    print(f"No hay fotos en {TEST_CAMARA}/<Color>/ — paso omitido.")

## 14. Curvas del entrenamiento (opcional)

In [ ]:
curvas = sorted(glob.glob(f"{RUNS}/etapa*/results.csv"))
if curvas:
    fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
    for ruta in curvas:
        r = pd.read_csv(ruta)
        r.columns = [c.strip() for c in r.columns]
        etapa = ruta.split("/")[-2]
        for col, ax, est in (("train/loss", ejes[0], "--"), ("val/loss", ejes[0], "-")):
            if col in r:
                ax.plot(r["epoch"], r[col], est, label=f"{etapa} {col}")
        if "metrics/accuracy_top1" in r:
            ejes[1].plot(r["epoch"], r["metrics/accuracy_top1"], label=etapa)
    ejes[0].set_title("Pérdida (si val sube mientras train baja: sobreajuste)"); ejes[0].legend(fontsize=8)
    ejes[1].set_title("Acierto top-1 en val"); ejes[1].legend(fontsize=8)
    for ax in ejes:
        ax.set_xlabel("época")
    guardar_figura("curvas_entrenamiento.png")
else:
    print(f"No hay {RUNS}/etapa*/results.csv — paso omitido.")

## 15. Resumen con semáforo
Los umbrales son **orientativos**: sirven para saber qué revisar primero.

In [ ]:
def semaforo(valor, verde, amarillo, mayor_es_mejor=False):
    if valor is None or (isinstance(valor, float) and np.isnan(valor)):
        return "  -  "
    if mayor_es_mejor:
        return "VERDE" if valor >= verde else "AMARILLO" if valor >= amarillo else "ROJO"
    return "VERDE" if valor <= verde else "AMARILLO" if valor <= amarillo else "ROJO"

azar = RESUMEN["azar_pct"]
caida = (RESUMEN["test_acierto"] - RESUMEN["test_acierto_limpio"]) if RESUMEN.get("test_acierto_limpio") is not None else None
lineas = [
    ("Fotos en total", RESUMEN["fotos_total"], "", ""),
    ("Desbalance entre colores (x)", RESUMEN["desbalance_max_min"], semaforo(RESUMEN["desbalance_max_min"], 1.5, 2.5), "ideal < 1.5x"),
    ("Acierto en val (%)", RESUMEN["val_acierto"], "", ""),
    ("Acierto en test (%)", RESUMEN["test_acierto"], "", ""),
    ("Test con casi duplicados de train (%)", RESUMEN["casi_dup_test_pct"], semaforo(RESUMEN["casi_dup_test_pct"], 5, 15), "fuga de datos"),
    ("Acierto en test limpio (%)", RESUMEN.get("test_acierto_limpio"), "", "la cifra más honesta del dataset"),
    ("Caída test -> test limpio (puntos)", None if caida is None else round(caida, 1), semaforo(caida, 2, 5), "cuánto infla la fuga"),
    ("Adivinar solo con el fondo (%)", RESUMEN["atajo_fondo_acierto"], semaforo(RESUMEN["atajo_fondo_acierto"], azar + 15, azar + 30), f"azar = {azar}%"),
    ("Modelo con el objeto tapado (%)", RESUMEN["solo_fondo_acierto"], semaforo(RESUMEN["solo_fondo_acierto"], azar + 15, azar + 30), f"azar = {azar}%"),
    ("Modelo solo con el centro (%)", RESUMEN["solo_centro_acierto"], "", "debería ser alto"),
    ("Grad-CAM: foco en el centro (%)", RESUMEN["gradcam_foco_centro_pct"], semaforo(RESUMEN["gradcam_foco_centro_pct"], 45, 30, True), "uniforme = 25%"),
    ("Confianza máx. sin objeto (%)", RESUMEN["sin_objeto_max_conf"], semaforo(RESUMEN["sin_objeto_max_conf"], 60, 70), "> 70% pasa el plan B"),
    ("Acierto con tu cámara (%)", RESUMEN.get("camara_acierto"), semaforo(RESUMEN.get("camara_acierto"), 85, 70, True), "examen real"),
]
res = pd.DataFrame(lineas, columns=["indicador", "valor", "semáforo", "nota"]).set_index("indicador")
display(res)
res.to_csv(f"{REV}/resumen.csv")
with open(f"{REV}/resumen.json", "w") as f:
    json.dump(RESUMEN, f, indent=2, ensure_ascii=False)
print("\nArchivos guardados en", REV)
for f in sorted(os.listdir(REV)):
    print("  ", f)